In [5]:
import sys, pathlib
BEDL = pathlib.Path("/project/6004619/dcs01/PyLDL/bedl_experiments")
if str(BEDL) not in sys.path:
    sys.path.insert(0, str(BEDL))

In [6]:

import importlib
import helpers

importlib.reload(helpers)
from helpers import read_results, dataframe_to_latex_table
import pandas as pd
import numpy as np

E0000 00:00:1779392611.229193 2595233 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779392611.662791 2595233 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779392614.212465 2595233 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779392614.212505 2595233 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779392614.212508 2595233 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779392614.212509 2595233 computation_placer.cc:177] computation placer already registered. Please check linka

In [7]:
def combine_folds(model, dataset, metrics):
    
    records = []
    for fold in list(range(10)):
        results = read_results(model, dataset, fold)
        record = {
            'model': model,
            'dataset': dataset,
            'fold': fold
        }
        full_metrics = []
        for metric in metrics:
            if type(results.get(f'{metric}_mean')) is list:
                record_list = results[f'{metric}_mean']
                for i, value in enumerate(record_list):
                    record[f'{metric}_{i}_mean'] = value
                    record[f'{metric}_{i}_var'] = results[f'{metric}_var'][i]
                    full_metrics.append(f'{metric}_{i}')

            else:
                record[f'{metric}_mean'] = results[f'{metric}_mean']
                record[f'{metric}_var'] = results[f'{metric}_var']
                record[f'{metric}_std'] = results[f'{metric}_var']**0.5
                full_metrics.append(metric)
        records.append(record)

    all_metrics_df = pd.DataFrame.from_records(records)
    combined = {
        'model': model,
        'dataset': dataset,
    }
    for metric in full_metrics:
        combined[f'{metric}_mean'] = all_metrics_df[f'{metric}_mean'].mean()
        combined[f'{metric}_err'] = all_metrics_df[f'{metric}_mean'].std() / (10 ** 0.5)
        metric_var = all_metrics_df[f'{metric}_var'].mean() + all_metrics_df[f'{metric}_mean'].var(ddof=0)
        combined[f'{metric}_std'] = metric_var**0.5
    
    return combined

def combine_metrics(models, datasets,
                    metrics, compact=False):

    all_results = []
    for model in models:
        for dataset in datasets:
            fold_results = combine_folds(model, dataset, metrics)
            all_results.append(fold_results)

    df = pd.DataFrame.from_records(all_results)
    df.sort_values(metrics[0] + '_mean', inplace=True)
    if compact:
        df = compact_metrics(df)
    return df

def compact_metrics(df):
    compacted = []
    for _, row in df.iterrows():
        record = {
            'model': row['model'],
            'dataset': row['dataset']
        }
        for col in df.columns:
            if col not in ['model', 'dataset']:
                if col.endswith('_mean'):
                    metric_name = col.replace('_mean', '')
                    err_col = f'{metric_name}_err'
                    if err_col in df.columns:
                        record[metric_name] = f"{row[col]:.3f}±{row[err_col]:.3f}"
                    else:
                        record[metric_name] = f"{row[col]:.3f}"
                elif not col.endswith('_err') and not col.endswith('_std'):
                    record[col] = row[col]
        compacted.append(record)
    return pd.DataFrame.from_records(compacted)


In [8]:
models = ['multiBelief', 'EDL', 'BEDL', 'EDL_BAYES', 'BEDL_BAYES',
          'BOOJUM', 'BOOJUM_BAYES', 'AA_BP', 'SNEFY_LDL']
datasets = ['SJAFFE']
metrics = ['kl_divergence', 'chebyshev', 'clark', 'canberra']
combined_results = combine_metrics(models, datasets, metrics, compact=True)
combined_results.drop(columns=['dataset'], inplace=True)

dataframe_to_latex_table(df=combined_results, output_file='sjaffe_accuracy.tex',
                         caption='Accuracy for SJAFFE dataset', label='tab:sjaffe_accuracy'
                         )
combined_results

LaTeX table saved to: sjaffe_accuracy.tex


,model,kl_divergence,chebyshev,clark,canberra
0,multiBelief,0.043±0.003,0.090±0.003,0.325±0.009,0.664±0.019
1,BOOJUM,0.056±0.004,0.103±0.004,0.366±0.012,0.762±0.026
2,BEDL_BAYES,0.056±0.004,0.102±0.004,0.364±0.012,0.756±0.027
3,BOOJUM_BAYES,0.071±0.005,0.117±0.005,0.417±0.011,0.871±0.027
4,EDL_BAYES,0.073±0.004,0.120±0.005,0.425±0.012,0.888±0.026
5,AA_BP,0.074±0.004,0.120±0.005,0.426±0.012,0.890±0.026
6,EDL,0.074±0.005,0.120±0.005,0.423±0.012,0.889±0.026
7,BEDL,0.080±0.005,0.125±0.005,0.430±0.012,0.909±0.026
8,SNEFY_LDL,0.150±0.018,0.147±0.005,0.599±0.032,1.249±0.068


In [9]:
combined_results = combine_metrics(models, datasets, ['kl_divergence', 'per_label_fsc'], compact=True)
combined_results.drop(columns=['kl_divergence'], inplace=True)
combined_results.rename(columns={f'per_label_fsc_{i}': f'label {i}' for i in range(len(combined_results.columns) - 1)}, inplace=True)
combined_results.drop(columns=['dataset'], inplace=True)

dataframe_to_latex_table(df=combined_results, output_file='sjaffe_per_label_fsc.tex',
                         caption='Per-label FSC for SJAFFE dataset', label='tab:sjaffe_per_label_fsc'
                         )
combined_results


LaTeX table saved to: sjaffe_per_label_fsc.tex


,model,label 0,label 1,label 2,label 3,label 4,label 5
0,multiBelief,0.885±0.021,0.909±0.017,0.882±0.022,0.909±0.020,0.895±0.015,0.933±0.015
1,BOOJUM,0.882±0.016,0.887±0.022,0.881±0.021,0.895±0.023,0.894±0.024,0.914±0.026
2,BEDL_BAYES,0.876±0.012,0.910±0.019,0.885±0.015,0.909±0.023,0.899±0.019,0.912±0.025
3,BOOJUM_BAYES,0.906±0.020,0.907±0.018,0.899±0.018,0.895±0.025,0.902±0.026,0.914±0.024
4,EDL_BAYES,0.909±0.021,0.907±0.018,0.898±0.018,0.892±0.029,0.912±0.025,0.911±0.028
5,AA_BP,0.907±0.016,0.890±0.022,0.900±0.017,0.885±0.032,0.904±0.023,0.904±0.022
6,EDL,0.909±0.018,0.906±0.019,0.898±0.018,0.892±0.028,0.912±0.025,0.897±0.031
7,BEDL,0.911±0.018,0.907±0.019,0.899±0.017,0.888±0.030,0.918±0.022,0.876±0.025
8,SNEFY_LDL,0.912±0.016,0.875±0.018,0.896±0.016,0.909±0.020,0.906±0.014,0.906±0.015


In [10]:
def per_bin_fsc_stats(models, datasets, order_by='per_bin_fsc_min'):
    records = []
    for model in models:
        for dataset in datasets:
            stats = {
                "model": model,
                "dataset": dataset
            }
            all_folds = []
            for fold in list(range(10)):
                results = read_results(model, dataset, fold)
                all_folds.append(np.array(results['per_bin_fsc_mean']))
            all_folds = np.array(all_folds)
            fold_avg = all_folds.mean(axis=0)
            stats['per_bin_fsc_min'] = fold_avg.min()
            stats['per_bin_fsc_max'] = fold_avg.max()
            stats['per_bin_fsc_mean'] = fold_avg.mean()
            records.append(stats)
    
    df = pd.DataFrame.from_records(records)
    return df.sort_values(by=order_by, ascending=False)

per_bin = per_bin_fsc_stats(models, datasets)
per_bin

,model,dataset,per_bin_fsc_min,per_bin_fsc_max,per_bin_fsc_mean
8,SNEFY_LDL,SJAFFE,0.840000,0.940000,0.901065
7,AA_BP,SJAFFE,0.826667,0.938889,0.897454
3,EDL_BAYES,SJAFFE,0.822222,0.975556,0.905056
2,BEDL,SJAFFE,0.805556,0.966667,0.902361
5,BOOJUM,SJAFFE,0.783333,0.974444,0.894213
1,EDL,SJAFFE,0.783333,0.966667,0.904259
4,BEDL_BAYES,SJAFFE,0.777778,0.964444,0.899907
6,BOOJUM_BAYES,SJAFFE,0.777778,0.966667,0.906111
0,multiBelief,SJAFFE,0.766667,0.993333,0.903519


In [10]:
dataframe_to_latex_table(df=per_bin, output_file='per_bin_sjaffe_conformal.tex', round_float=3)

LaTeX table saved to: per_bin_sjaffe_conformal.tex
